In [0]:
# ---------------------------------------------
# SILVER LAYER: LOAD BRONZE DATA
# ---------------------------------------------
# Purpose:
# This step reads the raw data stored in the Bronze Delta table
# into a Spark DataFrame for transformation and cleaning.
#
# Why this is needed:
# - Acts as the input for all Silver layer transformations
# - Ensures separation between raw storage (Bronze) and processing layer (Silver)
# - Enables reusable and modular pipeline design

df= spark.table("rides_catalog.bronze.rides_raw")

In [0]:
# ---------------------------------------------
# SILVER LAYER: DATA CLEANING & VALIDATION
# ---------------------------------------------
# Purpose:
# This step applies data quality rules and business validations
# to transform raw Bronze data into a clean Silver dataset.
#
# Why this is needed:
# - Ensures only valid and consistent records are used for analytics
# - Removes duplicates and invalid records
# - Applies business rules before aggregation in Gold layer
#
# Data Quality Rules Applied:
# 1. Remove records with missing driver_id or user_id
# 2. Filter out invalid fare values (fare_amount > 0)
# 3. Remove duplicate events using event_id
# 4. Keep only valid event types for analysis


from pyspark.sql.functions import col

df_silver = (
    df
    .filter(col("driver_id").isNotNull() & col("user_id").isNotNull())
    .filter(col("fare_amount") > 0)
    .dropDuplicates(["event_id"])
    .filter(col("event_type").isin("complete", "accept", "request", "cancel", "start"))
)

In [0]:
%sql
-- ---------------------------------------------
-- SILVER LAYER: SCHEMA CREATION
-- ---------------------------------------------
-- Purpose:
-- This schema represents the Silver layer in the Medallion Architecture.
-- It stores cleaned, validated, and enriched data derived from the Bronze layer.
--
-- Why this is needed:
-- - Separates raw data (Bronze) from cleaned data (Silver)
-- - Enables structured transformation pipeline (Bronze → Silver → Gold)
-- - Supports governance and data organization within Unity Catalog
--
-- Data Characteristics:
-- - Cleaned and validated records
-- - Deduplicated data
-- - Business rules applied (e.g., valid fares, event filtering)

create schema if not exists rides_catalog.silver;

In [0]:
# ---------------------------------------------
# SILVER LAYER: WRITE CLEANED DATA TO DELTA TABLE
# ---------------------------------------------
# Purpose:
# This step persists the cleaned and validated Silver dataset
# into a managed Delta table within Unity Catalog.
#
# Why this is needed:
# - Stores transformed data in a reliable, queryable format
# - Enables downstream Gold layer aggregations
# - Supports ACID transactions and data versioning via Delta Lake
#
# Notes:
# - Data is already cleaned (nulls removed, duplicates dropped, rules applied)
# - Stored in Silver schema as part of Medallion Architecture

df_silver.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("rides_catalog.silver.rides_silver")